In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

spark = SparkSession.builder.appName("NYC Taxi Data").getOrCreate()

df = spark.read.table("nyc_silver.yellow_taxi_silver.yellow_taxi_transformed")

# monthly average trip
monthly_avg_trip_distance = df.groupBy(
        f.date_trunc("month", f.col("tpep_pickup_datetime")).alias("month"),
        "vendor_id"
    ).agg(
        f.count("*").alias("total_monthly_trips"),
        f.round(f.sum("trip_distance"), 2).alias("total_monthly_trip_distance"),
        f.round(f.avg("trip_distance"), 2).alias("monthly_avg_trip_distance")
    ).orderBy("month", "vendor_id")

# weekly average trip
weekly_avg_trip_distance = df.groupBy(
  f.date_trunc("week", f.col("tpep_pickup_datetime")).alias("week"),
  "vendor_id"
).agg(
  f.count("*").alias("total_weekly_trips"),
  f.round(f.sum("trip_distance"), 2).alias("total_weekly_trip_distance"),
  f.round(f.avg(f.col("trip_distance")),2).alias("weekly_avg_trip_distance")
).orderBy("week", "vendor_id")

# daily average trip
daily_avg_trip_distance = df.groupBy(
  f.date_trunc("day", f.col("tpep_pickup_datetime")).alias("day"),
  "vendor_id"
).agg(
  f.count("*").alias("total_daily_trips"),
  f.round(f.sum("trip_distance"), 2).alias("total_daily_trip_distance"),
  f.round(f.avg(f.col("trip_distance")), 2).alias("daily_avg_trip_distance")
).orderBy("day", "vendor_id")

# monthly revenue
monthly_revenue_by_vendor = df.groupBy(
    f.date_trunc("month", f.col("tpep_pickup_datetime")).alias("month"),
    "vendor_id"
  ).agg(
    f.count("*").alias("trip_count"),
    f.round(f.sum(f.col("total_amount")), 2).alias("monthly_revenue"),
    f.round(f.avg(f.col("fare_amount")), 2).alias("average_fare_amount")
  ).orderBy("month", "vendor_id")

# weekly revenue
weekly_revenue_by_vendor = df.groupBy(
    f.date_trunc("week", f.col("tpep_pickup_datetime")).alias("week"),
    "vendor_id"
  ).agg(
    f.count("*").alias("trip_count"),
    f.round(f.sum(f.col("total_amount")), 2).alias("weekly_revenue"),
    f.round(f.avg(f.col("fare_amount")), 2).alias("average_fare_amount")
  ).orderBy("week", "vendor_id")

# daily revenue
daily_revenue_by_vendor = df.groupBy(
    f.to_date("tpep_pickup_datetime").alias("trip_date"),
    "vendor_id"
).agg(
    f.count("*").alias("trip_count"),
    f.round(f.sum("total_amount"), 2).alias("daily_revenue"),
    f.round(f.avg("fare_amount"), 2).alias("average_fare")
).orderBy("trip_date", "vendor_id")

# top pickup location
top_pickup_locations = df.groupBy("pulocation_id").agg(
    f.count("*").alias("trip_count"),
    f.round(f.sum("total_amount"), 2).alias("total_revenue"),
    f.round(f.avg("trip_distance"), 2).alias("avg_trip_distance")
).orderBy(f.desc("trip_count"))

# monthly, weekly and daily average trip distance by vendor
monthly_avg_trip_distance.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("nyc_gold.yellow_taxi_gold.monthly_avg_trip_distance_by_vendor")
weekly_avg_trip_distance.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("nyc_gold.yellow_taxi_gold.weekly_avg_trip_distance_by_vendor")
daily_avg_trip_distance.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("nyc_gold.yellow_taxi_gold.daily_avg_trip_distance_by_vendor")

# monhtly, weekly and daily revenue by vendor
monthly_revenue_by_vendor.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("nyc_gold.yellow_taxi_gold.monthly_revenue_by_vendor")
weekly_revenue_by_vendor.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("nyc_gold.yellow_taxi_gold.weekly_revenue_by_vendor")
daily_revenue_by_vendor.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("nyc_gold.yellow_taxi_gold.daily_revenue_by_vendor")

# top pickup location
top_pickup_locations.write.mode("overwrite").saveAsTable("nyc_gold.yellow_taxi_gold.top_pickup_locations")